In [ ]:


import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date
import yaml

PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_SP = Path.home()/r'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Vibrant and Inclusive Places\People and Community\Pop and Demographics\Pop_6 Birth Rates'
PATH_CONFIG0 = Path.home()/r'Documents\Projects\Regional-Monitoring\Indicator_Gen\config'


EXPORT=True
ABOUT=True
UPDATE=False


EST='ACS5'
GEO='National'



def clean_fips(df):
        
    '''
    FIPS codes are often used across data sources but they don't always come in the same format, especially when using different file types (.csv, .xlsx, ...)
    This function standardizes the FIPS format for various FIPS codes
    '''

    print('Cleaning FIPS codes to standard format...')
    
    if 'STATEFP' in df.columns:                          df['STATEFP'                         ] = df['STATEFP'                         ].astype(str).apply('{:0>2}'.format)
    if 'State FIPS' in df.columns:                       df['State FIPS'                      ] = df['State FIPS'                      ].astype(str).apply('{:0>2}'.format)
    if 'Place ID' in df.columns:                         df['Place ID'                        ] = df['Place ID'                        ].astype(str).apply('{:0>5}'.format)
    if 'COUNTYFP' in df.columns:                         df['COUNTYFP'                        ] = df['COUNTYFP'                        ].astype(str).apply('{:0>3}'.format)
    if 'County FIPS' in df.columns:                      df['County FIPS'                     ] = df['County FIPS'                     ].astype(str).apply('{:0>3}'.format)
    if 'Congressional District' in df.columns:           df['Congressional District'          ] = df['Congressional District'          ].astype(str).apply('{:0>2}'.format)
    if 'State Legislative Upper District' in df.columns: df['State Legislative Upper District'] = df['State Legislative Upper District'].astype(str).apply('{:0>3}'.format)
    if 'State Legislative Lower District' in df.columns: df['State Legislative Lower District'] = df['State Legislative Lower District'].astype(str).apply('{:0>3}'.format)

    return df




# Writes about page for each indicator
def write_about(df_pop6):

    '''
    User defined function to create/export about documentation for each indicator
    Inputs: yaml file, specific indicator inputs (geography, sample type, ...), data frame to export, file paths, ...
    Uses user defined inputs to organize .yaml file subset into pandas data frame then exports to excel file sheet
    '''

    # Reads in .yaml file
    # Defines initialized objects in the yaml file with objects defined in processing script

    path_yaml = PATH_CONFIG0 / 'about.yaml'
    
    try:
        with open(path_yaml, 'r') as yaml_file:
            yaml_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    except FileNotFoundError:
        print(f"Error: The file at {path_yaml} does not exist.")
    except Exception as e:
        print(f"An error occurred: {e}")

    df = pd.DataFrame.from_dict([yaml_about[EST]['ACS']['Pop_6']]).T.reset_index().rename(columns = {'index': 'Indicator', 0: 'Pop_6'})

    df.loc[df['Indicator'] == 'Last Updated', 'Pop_6'] = date.today().strftime('%Y-%m-%d')
    df.loc[df['Indicator'] == 'Year(s)'     , 'Pop_6'] = f"{df_pop6['Year'].min()}-{df_pop6['Year'].max()}"
    df.loc[df['Indicator'] == 'Geography', 'Pop_6'] = GEO

    if GEO == 'Places':
        df.loc[df['Indicator'] == 'Geography', 'Pop_6'] = 'Census Designated Places (Jurisdictions)'

    # Split notes into rows, for visual clarity in about
    # Find the row with 'Notes', then use that to take the information
    # Separate based off of NewLines, make the rows with this
    # Make a blank row past the first one. This way, we don't have to see notes as a cell like 7 times.
    # Create a df from the new separated rows. Drop the old notes row
    # Combine original with new rows
    # Finally, we split the notes
    def split_notes(df):
        row_notes = df[df['Indicator'] == 'Notes'].copy()
        notes = row_notes['Pop_6'].values[0]
        
        lines = notes.split('\\n')
        rows_new = [{'Indicator': 'Notes' if i == 0 else '', 'Pop_6': line} for i, line in enumerate(lines) if line]
        
        df_new = pd.DataFrame(rows_new)
        df_filtered = df[df['Indicator'] != 'Notes']
       
        df_notes = pd.concat([df_filtered, df_new], ignore_index=True)
        
        return df_notes

    df = split_notes(df)

    return df

   
def post_pop6(df, geography, estimate):

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    path_weights = Path.home()/r'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data' / 'Reference' / 'Weights'
    file_weights = path_weights / f'Total_Population {geography} {estimate}.xlsx'
    df_pop = pd.read_excel(file_weights, sheet_name=geography)

    df_pop = df_pop.rename(columns={'Population': f'Total Population', 'MSA_ID':'MSA ID'})

    conditions = [
                    (df_pop["Race/Ethnicity"] == 'All'                                            ),
                    (df_pop["Race/Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                    (df_pop["Race/Ethnicity"] == 'Asian (NH)'                                     ),
                    (df_pop["Race/Ethnicity"] == 'Black or African American (NH)'                 ),
                    (df_pop["Race/Ethnicity"] == 'Hispanic or Latino'                             ),
                    (df_pop["Race/Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                    (df_pop["Race/Ethnicity"] == 'White (NH)'                                     ),
                    (df_pop["Race/Ethnicity"] == 'Some other race (NH)'                           ),
                    (df_pop["Race/Ethnicity"] == 'Two or more races (NH)'                         )
                ]
    choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino", "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
    df_pop["Race/Ethnicity"] = np.select(conditions, choices)

    if geography == 'MSA': field_id = 'MSA ID'
    elif geography == 'MPO': field_id = 'MPO'
    else: field_id ='NAME'

    if 'Total Population' in df.columns: df = df.drop('Total Population', axis=1)

    df = df.merge(df_pop[[field_id, 'Year', 'Race/Ethnicity', 'Total Population']], on=[field_id, 'Year', 'Race/Ethnicity'], how='left')

    df['Total Population'] = df['Total Population'].fillna(0)
    df['Total Population'] = df['Total Population'].replace(0, 1)
    df['Birth Rate Per 1,000 People'] = (df['Population'] / df['Total Population']) * 1000
    df = df[df['Variable'] == 'Woman aged 15-44 who had a birth in the past 12 months']
    
    return df



geographies=['Block Groups', 'Tracts', 'Counties', 'MSA', 'States', 'National',
             'Places', 'Congressional Districts', 'State Legislative Lower Districts', 'State Legislative Upper Districts']


if GEO in ['Congressional Districts', 'State Legislative Lower Districts', 'State Legislative Upper Districts']:
    if GEO == 'Congressional Districts': sheet_name='CD'
    if GEO == 'State Legislative Lower Districts': sheet_name='SLDL'
    if GEO == 'State Legislative Upper Districts': sheet_name='SLDU'
else:
    sheet_name=GEO

file_in = PATH_SP / f'Pop_6 {GEO} {EST}.xlsx'
df=pd.read_excel(file_in, sheet_name=sheet_name)
df = df.rename(columns={'Race_Ethnicity':'Race/Ethnicity'})

if GEO == 'MSA':
    df = df[df['MSA'].str.contains('Sacramento|Yuba')].reset_index(drop=True)
df = clean_fips(df)
df = post_pop6(df, geography=GEO, estimate=EST)

display(df.head())
df_about = write_about(df)
display(df_about)

if EXPORT:

    for path_ in [PATH_SERVER, PATH_SP]:
        file_out = path_/f'Pop_6 {GEO} {EST}.xlsx'

        if ABOUT:
            with pd.ExcelWriter(file_out, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                df_about.to_excel(writer, index=False, sheet_name='About', header=False)
                df.to_excel(writer, index=False, sheet_name=sheet_name)
        else:
            with pd.ExcelWriter(file_out, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                df.to_excel(writer, index=False, sheet_name=sheet_name)
        
        if ABOUT:
            if UPDATE:
                file_about = PATH_SP / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'
                with pd.ExcelWriter(file_about, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                    df_about.to_excel(writer, index=False, sheet_name='Pop_6', header=False)


